# DeepSeek Coder 6.7B (Q4) trên Colab T4 bằng Vulkan

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từ trên xuống. Notebook cài Nim 2.2.10 và Vulkan, clone repo kèm submodule, tải `model.nimq` cùng tokenizer từ [bobbydeveloper2014/deepseek-coder-6.7b-nimq](https://huggingface.co/bobbydeveloper2014/deepseek-coder-6.7b-nimq), build `nim_inference`, rồi chạy inference trên Vulkan. Tokenizer và chat template chạy thuần Nim; không cần `transformers`.

Tensor Core của T4 được dùng giống Codestral: các GEMM prefill (Q/K/V/O, gate/up/down của cả 32 layer) chạy kernel `KHR_cooperative_matrix`. Kernel này giải nén Q4 thành FP16 theo từng tile và cộng dồn FP32. Bộ autotune kiểm tra parity với kernel tham chiếu ngay trên GPU, đo ở M=64, rồi mới chọn. Decode (1 token/bước) là GEMV bị giới hạn bởi băng thông, nên dùng kernel scalar như Codestral.

Sửa `PROMPT`, `MAX_NEW_TOKENS`, `TEMPERATURE` ở cell inference.

In [ ]:
# 1) Kiểm tra runtime GPU và cài các gói hệ thống, gồm Vulkan runtime và shader compiler (glslc).
import os, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), 'Notebook này cần chạy trên Google Colab.'
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True))
except Exception as e:
    raise RuntimeError('Chưa có GPU NVIDIA. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.') from e

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','xz-utils','pkg-config','time','libvulkan1','libvulkan-dev','vulkan-tools','glslc','libegl1','libglx0','libx11-6','libxext6','libx11-xcb1','libxcb1','libxcb-dri3-0','libxcb-present0','libxfixes3','libxrandr2','libdrm2','libexpat1','libwayland-client0'], check=True)

In [ ]:
# 2) Cài Nim 2.2.10 chính chủ và xác minh checksum (tokenizer thuần Nim).
import hashlib, urllib.request, tarfile, os, subprocess
from pathlib import Path

NIM_VERSION = '2.2.10'
TOOLS = Path('/content/tools'); TOOLS.mkdir(exist_ok=True)
archive = TOOLS / f'nim-{NIM_VERSION}-linux_x64.tar.xz'
url = f'https://nim-lang.org/download/nim-{NIM_VERSION}-linux_x64.tar.xz'

# Tải file với User-Agent giả lập trình duyệt để tránh lỗi 403
if not archive.exists():
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(archive, 'wb') as out_file:
        out_file.write(response.read())

# Xác minh checksum
expected = '0a3a38752e97e9d44aa479b3a7b37336dfe0176daf22ee5b5218ad0991ecd211'
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual == expected, f'Nim checksum mismatch: {actual}'

# Giải nén
with tarfile.open(archive, 'r:xz') as tf:
    tf.extractall(TOOLS, filter='data')

# Cấu hình PATH
NIM_HOME = TOOLS / f'nim-{NIM_VERSION}'
os.environ['PATH'] = f"{NIM_HOME}/bin:" + os.environ['PATH']

# Kiểm tra phiên bản
subprocess.run(['nim', '--version'], check=True)

In [ ]:
# 3) Clone repository bản main mới nhất và tải submodule đã ghim trong repo.
import os, subprocess
from pathlib import Path

URL = 'https://github.com/bobbyshop-vui/nimformers-framework.git'
BRANCH = 'main'
REPO = Path('/content/nimformers-framework')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'sync', '--recursive'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'], check=True)
os.chdir(REPO)
print('Repo:', REPO, '| branch:', BRANCH)
print(subprocess.check_output(['git', 'log', '--oneline', '-1'], text=True))

In [ ]:
# 4) Cài Vulkan + kiểm tra ICD NVIDIA và xác nhận dGPU (không chấp nhận lavapipe/CPU)
import subprocess, os, re
from pathlib import Path

# --- Bước 1: Cài vulkan-tools ---
print("[1/4] Cài vulkan-tools...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'vulkan-tools'], check=True)

# --- Bước 2: Xác định phiên bản driver NVIDIA và cài libnvidia-gl tương ứng ---
print("[2/4] Phát hiện phiên bản driver NVIDIA...")
smi = subprocess.check_output(['nvidia-smi'], text=True)
m = re.search(r'Driver Version:\s*(\d+)\.', smi)
assert m, "Không đọc được Driver Version từ nvidia-smi"
driver_major = m.group(1)
print(f"    -> Driver major: {driver_major}")

pkg = f'libnvidia-gl-{driver_major}'
print(f"[2/4] Cài {pkg}...")
subprocess.run(['apt-get', 'install', '-y', '-qq', pkg], check=True)

# --- Bước 3: Đảm bảo file ICD NVIDIA có thể được loader tìm thấy ---
print("[3/4] Cấu hình Vulkan ICD...")
Path('/usr/share/vulkan/icd.d').mkdir(parents=True, exist_ok=True)
candidates = [
    '/etc/vulkan/icd.d/nvidia_icd.json',
    '/usr/share/vulkan/icd.d/nvidia_icd.json',
    f'/usr/lib/x86_64-linux-gnu/libnvidia-gl-{driver_major}/nvidia_icd.json',
]
src = next((c for c in candidates if Path(c).exists()), None)
if src and src != '/usr/share/vulkan/icd.d/nvidia_icd.json':
    subprocess.run(['ln', '-sf', src, '/usr/share/vulkan/icd.d/nvidia_icd.json'], check=True)
    print(f"    -> symlink: {src} -> /usr/share/vulkan/icd.d/nvidia_icd.json")
else:
    print(f"    -> ICD có sẵn: {src}")

# --- Bước 4: Chạy kiểm tra ---
print("[4/4] Kiểm tra Vulkan...")
subprocess.run(['vulkaninfo', '--summary'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary = subprocess.check_output(['vulkaninfo', '--summary'],
                                   text=True, stderr=subprocess.STDOUT)
print(''.join(line for line in summary.splitlines()
              if any(k in line.lower() for k in ('device name', 'device type', 'driver name'))))

assert 'NVIDIA' in summary.upper(), \
    'Không thấy NVIDIA Vulkan ICD. Runtime GPU hiện tại không expose Vulkan graphics.'
assert 'DISCRETE_GPU' in summary.upper(), \
    'Không tìm thấy NVIDIA dGPU qua Vulkan; dừng để tránh fallback CPU/iGPU.'

print("✅ Vulkan OK: NVIDIA dGPU đã được expose.")

In [ ]:
# 5) Tải model Q4 (~4.7 GB), config và tokenizer. Hugging Face tự resume nếu cell bị ngắt.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download

MODELS = REPO / 'test/deepseek-inference/models'
MODELS.mkdir(parents=True, exist_ok=True)
MODEL_REPO = 'bobbydeveloper2014/deepseek-coder-6.7b-nimq'
MODEL_REV = '2bd1dc094b4c8963232632aa79cdc4ef884efda1'
for name in ['config.json', 'tokenizer.json', 'tokenizer_config.json', 'model.nimq']:
    hf_hub_download(MODEL_REPO, filename=name, revision=MODEL_REV, local_dir=str(MODELS))
for f in sorted(MODELS.iterdir()):
    print(f'{f.name:24} {f.stat().st_size / 2**20:10.1f} MB')

In [ ]:
# 6) Biên dịch shader Vulkan và binary nim_inference.
import os
os.chdir(str(REPO))

!make shaders
!make build-nim-inference

## Inference

`model.nimq` được mmap thẳng từ đĩa, không đọc cả file vào RAM. Trên T4 (dGPU) mỗi weight được copy một lần từ mmap vào VRAM. `BYBY_VULKAN_DEVICE_LOCAL=auto` đặt weights ở VRAM khi Vulkan nhận diện dGPU NVIDIA.

Lúc load, mỗi shape Q4 được autotune hai lần: M=1 cho decode và M=64 cho prefill. Dòng `[q4-tune:vulkan] ... M=64: KHR-cooperative-matrix` nghĩa là GEMM prefill của shape đó chạy trên Tensor Core. Dòng `[vulkan] ... matrix=KHR_cooperative_matrix FP16/FP32` xác nhận driver hỗ trợ.

`lm_head` chỉ tính logits cho token cuối của prompt. Lần chạy đầu phải biên dịch shader bằng glslc nên chậm hơn; các lần sau dùng cache.

In [ ]:
# 7) Chạy inference.
import os
from pathlib import Path

PROMPT = 'Write a Python function to return the n-th Fibonacci number.'
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0          # 0 = greedy
TOP_P = 0.9
model = MODELS / 'model.nimq'
assert model.exists() and (MODELS / 'tokenizer.json').exists()

os.chdir(str(REPO))

print('Backend Vulkan | mmap | max tokens =', MAX_NEW_TOKENS)
print('model =', model)

!BYBY_MODEL_MMAP=1 \
 BYBY_VULKAN_DEVICE_LOCAL=auto \
 BYBY_VULKAN_TENSOR_CORES=1 \
 ./nim_inference "{MODELS}" "{model}" vulkan "{PROMPT}" {MAX_NEW_TOKENS} {TEMPERATURE} {TOP_P}

### Tài liệu

- [Nim 2.2.10](https://nim-lang.org/download/)
- [Google Colab FAQ: GPU runtime](https://research.google.com/colaboratory/faq.html)
- [VK_KHR_cooperative_matrix](https://registry.khronos.org/vulkan/specs/latest/man/html/VK_KHR_cooperative_matrix.html)
- [DeepSeek Coder 6.7B NimQ trên Hugging Face](https://huggingface.co/bobbydeveloper2014/deepseek-coder-6.7b-nimq)